# UMI in MuJoCo - Phase 3: Generate demonstrations

In the real UMI pipeline a person walks around with a handheld gripper and a GoPro, and SLAM turns the videos into gripper poses. Here a **scripted expert** plays the human: it picks up the cup and places it on the saucer while the simulator records what a real UMI demo contains:

| recorded key | meaning | shape per episode |
|---|---|---|
| `camera0_rgb` | wrist camera image (stands in for the GoPro) | (T, 224, 224, 3) uint8 |
| `robot0_eef_pos` | gripper tip position in the world frame | (T, 3) |
| `robot0_eef_rot_axis_angle` | gripper orientation as axis-angle | (T, 3) |
| `robot0_gripper_width` | opening between the fingers, metres | (T, 1) |

(plus `cmd_eef_pos`, `cmd_gripper_width` and `timestamp`, kept for debugging.) Episodes are recorded at **10 Hz**. The simulator itself runs at 50 Hz.

**What the expert knows vs. what the policy will see:** the expert reads the true cup and saucer positions from the simulator. The policy trained in Phase 5 never gets those - only the wrist image, the end-effector pose and the gripper width, as with the real UMI hardware.

**How the expert moves:** hover over the cup, descend, close the gripper, lift, carry it over the saucer, lower, release, retreat. Gripper yaw, heights and speeds are randomised a little for every episode so the demos are not identical. Only episodes that end with the cup upright on the saucer are saved.

Runs on Colab's own Python (no Phase 1 environment needed). It reuses the scene code you saved to Drive in Phase 2.

## 1. Paths and Google Drive

In [ ]:
USE_DRIVE = True

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/umi_mujoco'
else:
    ROOT = '/content/umi_mujoco'

DEMO_DIR = f'{ROOT}/sim_demos'
os.makedirs(DEMO_DIR, exist_ok=True)
PY = sys.executable
os.environ.update(ROOT=ROOT, DEMO_DIR=DEMO_DIR, PY=PY)
print('demos will be saved in', DEMO_DIR)

## 2. Install MuJoCo and download the Panda model (about 1-2 minutes)

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y libegl1 libgl1 libgl1-mesa-dri libglfw3 libosmesa6-dev patchelf > /dev/null
"$PY" -m pip -q install mujoco imageio
"$PY" -c "import mujoco; print('mujoco', mujoco.__version__)"
MEN=/content/mujoco_menagerie
if [ ! -d "$MEN/franka_emika_panda" ]; then
  git clone --depth 1 --filter=blob:none --sparse https://github.com/google-deepmind/mujoco_menagerie.git "$MEN"
  cd "$MEN" && git sparse-checkout set franka_emika_panda
fi
ls "$MEN/franka_emika_panda" | head -3

## 3. Find a working OpenGL backend

In [ ]:
%%bash
cat > /content/gl_test.py <<'EOF'
import mujoco
m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size="1"/></worldbody></mujoco>')
d = mujoco.MjData(m)
r = mujoco.Renderer(m, 64, 64)
mujoco.mj_forward(m, d)
r.update_scene(d)
r.render()
print('ok')
EOF
rm -f /content/gl_backend.txt
for GL in egl osmesa; do
  if MUJOCO_GL=$GL PYOPENGL_PLATFORM=$GL "$PY" /content/gl_test.py > /dev/null 2>&1; then
    echo $GL > /content/gl_backend.txt
    break
  fi
done
if [ -f /content/gl_backend.txt ]; then echo "working OpenGL backend: $(cat /content/gl_backend.txt)"; else echo "NO WORKING GL BACKEND"; fi

## 4. Restore the Phase 2 scene code from Drive

In [ ]:
import shutil
src = f'{ROOT}/sim/umi_sim'
assert os.path.exists(f'{src}/scene.py') and os.path.exists(f'{src}/env.py'), (
    'Phase 2 code not found in ' + src + '. Re-run the Phase 2 notebook (its last cell saves the code to Drive).')
shutil.copytree(src, '/content/umi_sim', dirs_exist_ok=True)
print(sorted(os.listdir('/content/umi_sim')))

## 5. Write the expert and the collection script
- `expert.py` - the scripted pick-and-place (inverse kinematics + waypoints). **The tuning knobs are the constants at the top of this file** (`GRASP_Z`, `CLOSE_HOLD_S`, ...).
- `collect_demos.py` - runs many episodes, keeps the successful ones, writes one `ep_<seed>.npz` per demo. Safe to interrupt and re-run: it skips finished episodes.

In [ ]:
%%writefile /content/umi_sim/expert.py
"""Scripted expert for the cup-on-saucer task.

The expert may look at the true object poses (privileged information). The policy that is trained from its
demonstrations only ever sees the wrist camera, the end-effector pose and the gripper width, exactly like
UMI's real-world setup.

Joint targets come from damped-least-squares inverse kinematics on a private copy of the model, so they do not
depend on how well the simulated arm tracks them.
"""
import numpy as np
from scipy.spatial.transform import Rotation as R

from .scene import CUP_HALF_H, SAUCER_HALF_H

# --------------------------------------------------------------------------- tunable parameters
GRASP_Z = 0.035           # TCP height while grasping (the cup centre is at z = 0.03)
CLOSE_HOLD_S = 0.8        # wait this long for the fingers to close
OPEN_HOLD_S = 0.5         # wait this long for the fingers to open after placing
SETTLE_S = 0.6            # wait at the end so the objects come to rest before judging success
PLACE_CLEARANCE = 0.004   # the cup is released this far above the saucer
GRIPPER_OPEN = 0.08
GRIPPER_CLOSED = 0.0
MIN_LIFT_Z = 0.08         # the cup must be higher than this after lifting, otherwise the grasp failed

RECORD_KEYS = ("camera0_rgb", "robot0_eef_pos", "robot0_eef_rot_axis_angle", "robot0_gripper_width",
               "cmd_eef_pos", "cmd_gripper_width")


class MjKinematics:
    """Forward kinematics and Jacobian of the TCP site for the 7 arm joints, using a private MjData."""

    def __init__(self, model, tcp_site_id):
        import mujoco
        self.mj = mujoco
        self.model = model
        self.data = mujoco.MjData(model)
        hinge = int(mujoco.mjtJoint.mjJNT_HINGE)
        jids = [j for j in range(model.njnt) if int(model.jnt_type[j]) == hinge]
        if len(jids) != 7:
            raise RuntimeError("expected 7 hinge joints, found %d" % len(jids))
        self.qadr = np.array([model.jnt_qposadr[j] for j in jids])
        self.dadr = np.array([model.jnt_dofadr[j] for j in jids])
        self.lo = np.array(model.jnt_range[jids, 0], dtype=float)
        self.hi = np.array(model.jnt_range[jids, 1], dtype=float)
        self.site = tcp_site_id
        self._jp = np.zeros((3, model.nv))
        self._jr = np.zeros((3, model.nv))

    def fk(self, q):
        m, d = self.model, self.data
        d.qpos[self.qadr] = q
        self.mj.mj_kinematics(m, d)
        self.mj.mj_comPos(m, d)
        self.mj.mj_jacSite(m, d, self._jp, self._jr, self.site)
        pos = d.site_xpos[self.site].copy()
        rot = d.site_xmat[self.site].reshape(3, 3).copy()
        jac = np.vstack([self._jp[:, self.dadr], self._jr[:, self.dadr]])
        return pos, rot, jac


def ik_step(kin, q, pos_des, rot_des, q_home, iters=2, lam=0.05, max_dq=0.05, null_gain=0.1):
    """A few damped-least-squares iterations towards the desired TCP pose.
    The null-space term keeps the elbow close to the ready posture."""
    q = np.asarray(q, dtype=float).copy()
    for _ in range(iters):
        pos, rot, jac = kin.fk(q)
        err = np.concatenate([pos_des - pos, R.from_matrix(rot_des @ rot.T).as_rotvec()])
        jpinv = jac.T @ np.linalg.inv(jac @ jac.T + lam ** 2 * np.eye(6))
        dq = jpinv @ err + (np.eye(len(q)) - jpinv @ jac) @ (null_gain * (q_home - q))
        peak = np.abs(dq).max()
        if peak > max_dq:
            dq *= max_dq / peak
        q = np.clip(q + dq, kin.lo, kin.hi)
    return q


class ScriptedExpert:
    def __init__(self, env, rng=None, kin=None):
        self.env = env
        self.rng = rng if rng is not None else np.random.default_rng(0)
        self.kin = kin if kin is not None else MjKinematics(env.model, env.tcp_id)
        dt = getattr(env, "control_dt", None)
        self.dt = dt if dt else env.substeps * env.model.opt.timestep
        self.log = []

    # ------------------------------------------------------------------ main entry point
    def run(self, record_every=5, frame_hook=None):
        """Run one episode from the env's current (freshly reset) state.
        Returns {'success', 'reason', 'data', 'log', 'sim_time'}; 'data' holds the recorded arrays."""
        env = self.env
        self.record_every = int(record_every)
        self.frame_hook = frame_hook
        self.rec = {k: [] for k in RECORD_KEYS}
        self.log = []
        self.step_count = 0
        self.q_cmd = np.asarray(env.arm_q, dtype=float).copy()
        self.q_home = self.q_cmd.copy()
        pos0, self.R_home, _ = self.kin.fk(self.q_cmd)
        self.cmd_pos = pos0.copy()
        self.cmd_yaw = 0.0
        self.cmd_grip = GRIPPER_OPEN

        state = env.get_object_state()
        cup, saucer = state["cup_pos"].copy(), state["saucer_pos"].copy()
        rng = self.rng
        yaw = rng.uniform(-0.5, 0.5)           # the cup is round, so any gripper yaw works; it adds variety
        hover_z = rng.uniform(0.18, 0.24)
        lift_z = rng.uniform(0.16, 0.22)
        v_move = rng.uniform(0.18, 0.30)       # m/s
        v_down = rng.uniform(0.08, 0.12)
        v_lift = rng.uniform(0.12, 0.20)

        self._record()                          # observation at t = 0

        # 1. above the cup, gripper open
        self._move_to([cup[0], cup[1], hover_z], v_move, yaw, GRIPPER_OPEN)
        self._log("approach")
        # 2. down around the cup
        self._move_to([cup[0], cup[1], GRASP_Z], v_down, yaw, GRIPPER_OPEN)
        self._log("descend")
        # 3. close the gripper
        self._hold(CLOSE_HOLD_S, GRIPPER_CLOSED)
        self._log("close")
        # 4. lift
        self._move_to([cup[0], cup[1], lift_z], v_lift, yaw, GRIPPER_CLOSED)
        self._log("lift")
        st = env.get_object_state()
        if st["cup_pos"][2] < MIN_LIFT_Z:
            return self._result(False, "grasp_failed")
        tcp = env.eef_pose()[0]
        off = st["cup_pos"] - tcp               # where the cup sits relative to the TCP (it may have shifted)

        # 5. carry it above the saucer so that the CUP (not the TCP) ends up centred on the saucer
        px, py = saucer[0] - off[0] + rng.uniform(-0.008, 0.008), saucer[1] - off[1] + rng.uniform(-0.008, 0.008)
        self._move_to([px, py, lift_z], v_move, yaw, GRIPPER_CLOSED)
        self._log("carry")
        if env.get_object_state()["cup_pos"][2] < MIN_LIFT_Z:
            return self._result(False, "dropped_in_transit")
        # 6. lower until the cup bottom is just above the saucer
        place_z = 2 * SAUCER_HALF_H + CUP_HALF_H + PLACE_CLEARANCE - off[2]
        self._move_to([px, py, place_z], v_down, yaw, GRIPPER_CLOSED)
        self._log("lower")
        # 7. release, 8. retreat, 9. settle
        self._hold(OPEN_HOLD_S, GRIPPER_OPEN)
        self._log("release")
        self._move_to([px, py, lift_z], v_lift, yaw, GRIPPER_OPEN)
        self._log("retreat")
        self._hold(SETTLE_S, GRIPPER_OPEN)
        self._log("settle")

        st = env.get_object_state()
        dxy = float(np.linalg.norm(st["cup_pos"][:2] - st["saucer_pos"][:2]))
        dz = float(st["cup_pos"][2] - (2 * SAUCER_HALF_H + CUP_HALF_H))
        up = self._cup_up()
        width = float(env.gripper_width)
        ok = dxy < 0.02 and abs(dz) < 0.012 and up > 0.95 and width > 0.06
        reason = "success" if ok else "bad_final(dxy=%.3f dz=%.3f up=%.2f grip=%.3f)" % (dxy, dz, up, width)
        return self._result(ok, reason)

    # ------------------------------------------------------------------ motion primitives
    def _move_to(self, target, speed, yaw, grip):
        target = np.asarray(target, dtype=float)
        dist = float(np.linalg.norm(target - self.cmd_pos))
        duration = max(0.5, 1.875 * dist / speed)        # minimum-jerk peak speed is 1.875 x the mean speed
        self._segment(target, yaw, grip, duration)

    def _hold(self, duration, grip):
        self._segment(self.cmd_pos.copy(), self.cmd_yaw, grip, duration)

    def _segment(self, target, yaw_target, grip, duration):
        n = max(1, int(round(duration / self.dt)))
        p0, y0 = self.cmd_pos.copy(), self.cmd_yaw
        for k in range(1, n + 1):
            s = k / n
            a = s ** 3 * (10 - 15 * s + 6 * s * s)         # minimum-jerk time scaling
            pos = p0 + a * (target - p0)
            yaw = y0 + a * (yaw_target - y0)
            rot_des = R.from_euler("z", yaw).as_matrix() @ self.R_home
            self.q_cmd = ik_step(self.kin, self.q_cmd, pos, rot_des, self.q_home)
            self.cmd_pos, self.cmd_yaw, self.cmd_grip = pos, yaw, grip
            self.env.step(self.q_cmd, grip, render=False)
            self.step_count += 1
            if self.step_count % self.record_every == 0:
                self._record()
        self.cmd_pos, self.cmd_yaw = np.asarray(target, dtype=float), yaw_target

    # ------------------------------------------------------------------ bookkeeping
    def _record(self):
        obs = self.env.get_obs()
        rec = self.rec
        rec["camera0_rgb"].append(obs["camera0_rgb"])
        rec["robot0_eef_pos"].append(obs["robot0_eef_pos"])
        rec["robot0_eef_rot_axis_angle"].append(obs["robot0_eef_rot_axis_angle"])
        rec["robot0_gripper_width"].append(obs["robot0_gripper_width"])
        rec["cmd_eef_pos"].append(np.asarray(self.cmd_pos, dtype=np.float64))
        rec["cmd_gripper_width"].append(np.array([self.cmd_grip], dtype=np.float64))
        if self.frame_hook is not None:
            self.frame_hook(self.env)

    def _cup_up(self):
        """z component of the cup's up axis: 1.0 when upright, 0 when lying on its side."""
        e = self.env
        return float(e.data.xmat[e.cup_id].reshape(3, 3)[2, 2])

    def _log(self, name):
        tcp = self.env.eef_pose()[0]
        cup = self.env.get_object_state()["cup_pos"]
        self.log.append("%-8s t=%5.1fs  tcp_err=%.4f m  cup=(%.3f, %.3f, %.3f)  grip=%.3f" % (
            name, self.step_count * self.dt, float(np.linalg.norm(tcp - self.cmd_pos)),
            cup[0], cup[1], cup[2], float(self.env.gripper_width)))

    def _result(self, success, reason):
        data = {}
        for k, v in self.rec.items():
            arr = np.stack(v)
            data[k] = arr if k == "camera0_rgb" else arr.astype(np.float32)
        n = len(self.rec["camera0_rgb"])
        data["timestamp"] = (np.arange(n) * self.record_every * self.dt).astype(np.float32)
        return {"success": bool(success), "reason": reason, "data": data, "log": list(self.log),
                "sim_time": self.step_count * self.dt}

In [ ]:
%%writefile /content/umi_sim/collect_demos.py
"""Collect scripted-expert demonstrations in the MuJoCo scene.

Example:  python collect_demos.py --menagerie /content/mujoco_menagerie --out /content/drive/MyDrive/umi_mujoco/sim_demos --n 200
Safe to interrupt and re-run: finished episodes are skipped (files ep_<seed>.npz, plus failed_seeds.txt).
"""
import argparse, collections, json, os, sys, time
import numpy as np

sys.path.insert(0, "/content")
from umi_sim.scene import build_scene
from umi_sim.env import UmiSimEnv
from umi_sim.expert import ScriptedExpert


def save_gif(frames, path, fps=10):
    from PIL import Image
    imgs = [Image.fromarray(f) for f in frames]
    imgs[0].save(path, save_all=True, append_images=imgs[1:], duration=int(1000 / fps), loop=0)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--menagerie", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--n", type=int, default=10, help="number of successful demos wanted (or episodes to try with --no_save)")
    ap.add_argument("--start_seed", type=int, default=1000)
    ap.add_argument("--max_attempts", type=int, default=None)
    ap.add_argument("--record_every", type=int, default=5, help="control runs at 50 Hz, so 5 -> 10 Hz recordings")
    ap.add_argument("--img", type=int, default=224)
    ap.add_argument("--no_save", action="store_true", help="test run: do not write demos")
    ap.add_argument("--gif", default=None, help="write a GIF of the first episode of this run")
    ap.add_argument("--verbose", action="store_true", help="print the phase log of every episode")
    args = ap.parse_args()

    xml = build_scene(args.menagerie)
    env = UmiSimEnv(xml, img_size=args.img)
    record_dt = args.record_every * env.substeps * env.model.opt.timestep

    failed_path = os.path.join(args.out, "failed_seeds.txt")
    failed = set()
    saved = 0
    if not args.no_save:
        os.makedirs(args.out, exist_ok=True)
        if os.path.exists(failed_path):
            failed = {int(x) for x in open(failed_path).read().split()}
        saved = len([f for f in os.listdir(args.out) if f.startswith("ep_") and f.endswith(".npz")])
        print("found %d existing demos in %s" % (saved, args.out))

    def ep_path(s):
        return os.path.join(args.out, "ep_%05d.npz" % s)

    max_attempts = args.max_attempts or max(args.n * 3, 10)
    reasons = collections.Counter()
    attempts = successes = 0
    gif_pending = args.gif is not None
    seed = args.start_seed
    t_start = time.time()

    def keep_going():
        if args.no_save:
            return attempts < args.n
        return saved < args.n and attempts < max_attempts

    while keep_going():
        this, seed = seed, seed + 1
        if not args.no_save and (os.path.exists(ep_path(this)) or this in failed):
            continue
        attempts += 1
        env.reset(seed=this)
        frames, hook = [], None
        if gif_pending:
            def hook(e, frames=frames):
                frames.append(np.concatenate([e.render("front", 224, 298), e.render("wrist", 224, 224)], axis=1))
        t0 = time.time()
        expert = ScriptedExpert(env, rng=np.random.default_rng(this + 7919))
        ep = expert.run(record_every=args.record_every, frame_hook=hook)
        wall = time.time() - t0
        reasons[ep["reason"].split("(")[0]] += 1
        n_frames = len(ep["data"]["camera0_rgb"])
        print("seed %5d: %-14s %5.1f s sim | %3d frames | %.1f s wall" % (
            this, ep["reason"].split("(")[0], ep["sim_time"], n_frames, wall), flush=True)
        if args.verbose or not ep["success"]:
            for line in ep["log"]:
                print("      " + line)
            if not ep["success"]:
                print("      ->", ep["reason"])
        if gif_pending:
            save_gif(frames, args.gif)
            print("      wrote GIF:", args.gif)
            gif_pending = False

        if ep["success"]:
            successes += 1
            if not args.no_save:
                tmp = ep_path(this) + ".tmp"
                with open(tmp, "wb") as f:
                    np.savez_compressed(f, seed=this, record_dt=record_dt, **ep["data"])
                os.replace(tmp, ep_path(this))
                saved += 1
        elif not args.no_save:
            with open(failed_path, "a") as f:
                f.write("%d\n" % this)

    wall = time.time() - t_start
    print("\n%d episodes tried, %d succeeded (%.0f%%), %.1f s wall (%.1f s per episode)" % (
        attempts, successes, 100.0 * successes / max(attempts, 1), wall, wall / max(attempts, 1)))
    print("outcomes:", dict(reasons))
    if not args.no_save:
        print("demos in %s: %d" % (args.out, saved))
        with open(os.path.join(args.out, "last_run.json"), "w") as f:
            json.dump({"attempts": attempts, "successes": successes, "outcomes": dict(reasons),
                       "record_dt": record_dt, "total_demos": saved}, f, indent=1)
    env.close()


if __name__ == "__main__":
    main()

## 6. Watch one episode
Runs a single episode without saving anything, prints the phase-by-phase log and writes a GIF (left: side camera, right: what the wrist camera sees).

**Reading the log:** each line is the end of a phase. `tcp_err` is how far the real gripper is from where it was commanded - it should be a few millimetres (below about 0.01 m). `cup=(x, y, z)` is the cup position: z should be about 0.03 on the table, rise above 0.16 when lifted, and end near 0.04 on the saucer.

In [ ]:
%%bash
export MUJOCO_GL=$(cat /content/gl_backend.txt)
export PYOPENGL_PLATFORM=$MUJOCO_GL
export PYTHONPATH=/content
"$PY" /content/umi_sim/collect_demos.py --menagerie /content/mujoco_menagerie --out /content/test_out \
    --n 1 --start_seed 0 --no_save --verbose --gif /content/debug_episode.gif

In [ ]:
from IPython.display import Image, display
display(Image(filename='/content/debug_episode.gif'))

## 7. Test run: 10 episodes, nothing saved
**Go on to step 8 only if at least 9 of 10 succeed.** Failed episodes print their phase log and a reason:

| reason | what it means | what to try (constants at the top of `expert.py`) |
|---|---|---|
| `grasp_failed` | the cup was not lifted | raise `CLOSE_HOLD_S` to 1.2; try `GRASP_Z` 0.030 or 0.040 |
| `dropped_in_transit` | the cup slipped out while moving | lower the speeds (`v_move`, `v_lift` ranges in `run`) |
| `bad_final(...)` | the cup is not on the saucer / upright / gripper not open | read `dxy`, `dz`, `up`, `grip` in the message |

If you see something you cannot fix, paste the output of this cell to me.

In [ ]:
%%bash
export MUJOCO_GL=$(cat /content/gl_backend.txt)
export PYOPENGL_PLATFORM=$MUJOCO_GL
export PYTHONPATH=/content
"$PY" /content/umi_sim/collect_demos.py --menagerie /content/mujoco_menagerie --out /content/test_out \
    --n 10 --start_seed 100 --no_save 2>&1 | tail -40

## 8. Collect the demonstrations
Saves to Google Drive (`umi_mujoco/sim_demos`). Change `N_DEMOS` if you like: 100 is enough to see whether the pipeline works, 200-300 gives a better policy. A demo is about 17 s long (around 170 frames) and a few MB on disk, so 200 demos take roughly 0.5-1 GB of Drive space. Expect on the order of 10-30 minutes in total (rendering with `osmesa` is the slow case).

If Colab disconnects, reconnect, re-run steps 1-5 (skip 6-7) and then this cell again: it continues where it stopped.

In [ ]:
N_DEMOS = 200
os.environ['N_DEMOS'] = str(N_DEMOS)

In [ ]:
%%bash
export MUJOCO_GL=$(cat /content/gl_backend.txt)
export PYOPENGL_PLATFORM=$MUJOCO_GL
export PYTHONPATH=/content
"$PY" /content/umi_sim/collect_demos.py --menagerie /content/mujoco_menagerie --out "$DEMO_DIR" \
    --n "$N_DEMOS" --start_seed 1000 2>&1 | grep -v '^seed.*success' | tail -60

## 9. Inspect the dataset

In [ ]:
import glob, numpy as np
import matplotlib.pyplot as plt

files = sorted(glob.glob(f'{DEMO_DIR}/ep_*.npz'))
print(len(files), 'demos in', DEMO_DIR)
assert files, 'no demos found - run step 8 first'

lengths = []
for f in files:
    with np.load(f) as z:
        lengths.append(len(z['robot0_eef_pos']))
lengths = np.array(lengths)
print('frames per demo: min %d  mean %.0f  max %d  (10 Hz -> %.1f s on average)' % (
    lengths.min(), lengths.mean(), lengths.max(), lengths.mean() / 10))
print('total frames:', int(lengths.sum()))

z = np.load(files[0])
print()
for k in z.files:
    print('%-28s %-18s %s' % (k, z[k].shape, z[k].dtype))

In [ ]:
# wrist-camera filmstrip of the first demo
z = np.load(files[0])
idx = np.linspace(0, len(z['camera0_rgb']) - 1, 8).astype(int)
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for ax, i in zip(axes, idx):
    ax.imshow(z['camera0_rgb'][i]); ax.axis('off')
    ax.set_title('t=%.1fs  w=%.3f' % (z['timestamp'][i], z['robot0_gripper_width'][i, 0]), fontsize=9)
plt.show()

# gripper height and opening over time for 5 demos
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for f in files[:5]:
    d = np.load(f)
    axes[0].plot(d['timestamp'], d['robot0_eef_pos'][:, 2])
    axes[1].plot(d['timestamp'], d['robot0_gripper_width'][:, 0])
axes[0].set_title('gripper height (m)'); axes[0].set_xlabel('time (s)')
axes[1].set_title('gripper opening (m)'); axes[1].set_xlabel('time (s)')
plt.show()

## Phase 3 checklist

- [ ] Step 6: the GIF shows the robot picking up the cup and putting it on the saucer
- [ ] Step 7: at least 9 of 10 episodes succeeded
- [ ] Step 8: `N_DEMOS` demos saved in `umi_mujoco/sim_demos` on Drive
- [ ] Step 9: the filmstrip shows the wrist view with the cup and saucer, and the gripper opening goes open -> closed -> open

Next: Phase 4 - convert these episodes into UMI's zarr replay-buffer format (same keys and shapes as `cup_in_the_wild.zarr.zip`, which is why Phase 0 inspected it).